In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib, warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection   import train_test_split, cross_val_score, KFold
from sklearn.linear_model      import LinearRegression, Ridge, Lasso
from sklearn.ensemble          import RandomForestRegressor
from sklearn.preprocessing     import StandardScaler
from sklearn.metrics           import mean_absolute_error, mean_squared_error, r2_score
from xgboost                   import XGBRegressor

pd.set_option('display.float_format', '{:.4f}'.format)
print('All libraries loaded.')

All libraries loaded.


In [4]:
df = pd.read_csv('real_estate_cleaned.csv')
print(f'Shape: {df.shape}')
df.head(3)

Shape: (7925, 44)


,Price,Area,Location,No. of Bedrooms,Resale,MaintenanceStaff,Gymnasium,SwimmingPool,LandscapedGardens,JoggingTrack,...,GolfCourse,TV,DiningTable,Sofa,Wardrobe,Refrigerator,City,PricePerSqFt,AmenityScore,BedroomCategory
0,30000000,3340,JP Nagar Phase 1,4,0,1,1,1,1,1,...,0,0,0,0,0,0,Bangalore,8982.0400,14,4BHK+
1,7888000,1045,Dasarahalli on Tumkur Road,2,0,0,1,1,1,1,...,0,0,0,0,0,0,Bangalore,7548.3300,15,2BHK
2,4866000,1179,Kannur on Thanisandra Main Road,2,0,0,1,1,1,1,...,0,0,0,0,0,0,Bangalore,4127.2300,9,2BHK


In [5]:
TARGET    = 'Price'
DROP_COLS = ['Price', 'Location', 'City', 'PricePerSqFt', 'BedroomCategory']

location_means = df.groupby('Location')['Price'].mean().to_dict()
global_location_mean = float(df['Price'].mean())

df['Location_Encoded'] = df['Location'].map(location_means)

location_map = df.groupby('City')['Location'].unique().apply(list).to_dict()

feature_cols = [c for c in df.columns if c not in DROP_COLS]
X_base = df[feature_cols].copy()
y      = df[TARGET].copy()

city_dummies    = pd.get_dummies(df['City'], prefix='City', drop_first=True)
city_dummy_cols = city_dummies.columns.tolist()
X = pd.concat([X_base, city_dummies], axis=1)

FEATURE_NAMES = X.columns.tolist()
CITIES        = sorted(df['City'].unique().tolist())

BINARY_COLS = [
    'Resale', 'MaintenanceStaff', 'Gymnasium', 'SwimmingPool',
    'LandscapedGardens', 'JoggingTrack', 'RainWaterHarvesting', 'IndoorGames',
    'ShoppingMall', 'Intercom', 'SportsFacility', 'ATM', 'ClubHouse', 'School',
    '24X7Security', 'PowerBackup', 'CarParking', 'StaffQuarter', 'Cafeteria',
    'MultipurposeRoom', 'Hospital', 'WashingMachine', 'Gasconnection', 'AC',
    'Wifi', "Children'splayarea", 'LiftAvailable', 'BED', 'VaastuCompliant',
    'Microwave', 'GolfCourse', 'TV', 'DiningTable', 'Sofa', 'Wardrobe', 'Refrigerator'
]
BINARY_COLS = [c for c in BINARY_COLS if c in df.columns]

print(f'Features Setup Complete. Dimension: {X.shape}')

Features Setup Complete. Dimension: (7925, 45)


In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f'Train : {X_train.shape[0]:,} rows')
print(f'Test  : {X_test.shape[0]:,} rows')

Train : 6,340 rows
Test  : 1,585 rows


In [7]:
def evaluate(y_true, y_pred, label=''):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    r2   = r2_score(y_true, y_pred)
    mape = float(np.mean(np.abs((y_true - y_pred) / y_true)) * 100)
    if label:
        print(f'--- {label} ---')
        print(f'  R²   : {r2:.4f}')
        print(f'  MAE  : ₹{mae:,.0f}')
        print(f'  RMSE : ₹{rmse:,.0f}')
        print(f'  MAPE : {mape:.2f}%')
    return {'R2': round(r2,4), 'MAE': round(mae,0), 'RMSE': round(rmse,0), 'MAPE': round(mape,2),
            'train_size': int(X_train.shape[0]), 'test_size': int(X_test.shape[0])}

In [8]:
scaler       = StandardScaler()
X_train_sc   = scaler.fit_transform(X_train)
X_test_sc    = scaler.transform(X_test)
print('StandardScaler fitted. Shape:', X_train_sc.shape)

StandardScaler fitted. Shape: (6340, 45)


In [9]:
lr = LinearRegression()
lr.fit(X_train_sc, y_train)

y_pred_lr_train = lr.predict(X_train_sc)
y_pred_lr_test  = lr.predict(X_test_sc)

lr_train_metrics = evaluate(y_train, y_pred_lr_train, 'LR — TRAIN')
print()
lr_test_metrics  = evaluate(y_test,  y_pred_lr_test,  'LR — TEST')

--- LR — TRAIN ---
  R²   : 0.8543
  MAE  : ₹1,601,929
  RMSE : ₹2,395,473
  MAPE : 21.14%

--- LR — TEST ---
  R²   : 0.8313
  MAE  : ₹1,724,698
  RMSE : ₹2,595,761
  MAPE : 21.86%


In [ ]:
# Cross-validation for LR (needed for metadata)
cv_scores = cross_val_score(lr, X_train_sc, y_train, cv=5, scoring='r2')
print(f'LR CV R² : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

joblib.dump(lr,     'real_estate_model.joblib')
joblib.dump(scaler, 'real_estate_scaler.joblib')
joblib.dump({
    'feature_names'   : FEATURE_NAMES,
    'city_dummy_cols' : city_dummy_cols,
    'binary_cols'     : BINARY_COLS,
    'cities'          : CITIES,
    'metrics'         : lr_test_metrics,
    'amenity_score_max': int(df['AmenityScore'].max()),
    'area_range'      : (int(df['Area'].min()), int(df['Area'].max())),
    'price_range'     : (int(y.min()), int(y.max())),
    'model_type'      : 'LinearRegression',
    'cv_r2_mean'      : round(float(cv_scores.mean()), 4),
    'cv_r2_std'       : round(float(cv_scores.std()),  4),
}, 'model_metadata.joblib')

print('Exported:')
print('real_estate_model.joblib')
print('real_estate_scaler.joblib')
print('model_metadata.joblib')


LR CV R² : 0.8504 ± 0.0073
Exported (PRIMARY — Dash app files):
real_estate_model.joblib
real_estate_scaler.joblib
model_metadata.joblib


In [16]:
rf = RandomForestRegressor(
    n_estimators=1000,
    max_depth=15,
    min_samples_leaf=5,
    n_jobs=-1,
    random_state=42
)
rf.fit(X_train, y_train)

y_pred_rf_train = rf.predict(X_train)
y_pred_rf_test  = rf.predict(X_test)

rf_train_metrics = evaluate(y_train, y_pred_rf_train, 'RF — TRAIN')
print()
rf_test_metrics  = evaluate(y_test,  y_pred_rf_test,  'RF — TEST')

--- RF — TRAIN ---
  R²   : 0.9455
  MAE  : ₹849,400
  RMSE : ₹1,464,975
  MAPE : 9.72%

--- RF — TEST ---
  R²   : 0.8901
  MAE  : ₹1,205,477
  RMSE : ₹2,095,221
  MAPE : 13.55%


In [ ]:
joblib.dump(rf, 'rf_model.joblib')
joblib.dump({
    'feature_names'   : FEATURE_NAMES,
    'city_dummy_cols' : city_dummy_cols,
    'binary_cols'     : BINARY_COLS,
    'cities'          : CITIES,
    'metrics'         : rf_test_metrics,
    'model_type'      : 'RandomForestRegressor',
    'hyperparams'     : {'n_estimators':200,'max_depth':15,'min_samples_leaf':5},
    'area_range'      : (int(df['Area'].min()), int(df['Area'].max())),
    'price_range'     : (int(y.min()), int(y.max())),
}, 'rf_metadata.joblib')

print('Exported:')
print('rf_model.joblib')
print('rf_metadata.joblib')

In [43]:
xgb = XGBRegressor(
    n_estimators=400,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    verbosity=0
)
xgb.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=False
)

y_pred_xgb_train = xgb.predict(X_train)
y_pred_xgb_test  = xgb.predict(X_test)

xgb_train_metrics = evaluate(y_train, y_pred_xgb_train, 'XGB — TRAIN')
print()
xgb_test_metrics  = evaluate(y_test,  y_pred_xgb_test,  'XGB — TEST')

--- XGB — TRAIN ---
  R²   : 0.9777
  MAE  : ₹632,576
  RMSE : ₹936,799
  MAPE : 8.37%

--- XGB — TEST ---
  R²   : 0.9058
  MAE  : ₹1,121,762
  RMSE : ₹1,939,552
  MAPE : 12.66%


In [ ]:
joblib.dump(xgb, 'xgb_model.joblib')

metadata_dict = {
    'feature_names': FEATURE_NAMES,
    'binary_cols': BINARY_COLS,
    'city_dummy_cols': city_dummy_cols,
    'cities': CITIES, 
    'location_map': location_map,      
    'location_means': location_means,  
    'global_location_mean': global_location_mean, # Added fallback
    'amenity_score_max': int(df['AmenityScore'].max()),
    'area_range': (int(df['Area'].min()), int(df['Area'].max())),
    'price_range': (int(y.min()), int(y.max())),
    'model_type': 'XGBoost'
}

joblib.dump(metadata_dict, 'xgb_metadata.joblib')
print("Fully optimized artifacts exported successfully!")

✅ Fully optimized artifacts exported successfully!
